# SECOM 전처리 — 1주차

`01_eda.ipynb`에서 확인한 내용을 반영해 모델 학습용 데이터를 만든다.

1. 결측률 50% 초과 피처 제거
2. 남은 결측값 중앙값 대체
3. 분산 0(저분산) 피처 제거
4. `StandardScaler` 정규화
5. 클래스 불균형 처리(SMOTE) — 학습셋에만 적용
6. `data/processed/`에 저장

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

DATA_DIR = "../data/raw"
OUT_DIR = "../data/processed"
RANDOM_STATE = 42

## 1. 데이터 로드

In [2]:
X = pd.read_csv(f"{DATA_DIR}/secom.data", sep=" ", header=None)
X.columns = [f"feature_{i+1}" for i in range(X.shape[1])]

labels_raw = pd.read_csv(f"{DATA_DIR}/secom_labels.data", sep=" ", header=None, names=["label", "timestamp"])
y = (labels_raw["label"] == 1).astype(int)  # 1=불량(Fail), 0=정상(Pass)

print(X.shape, y.value_counts().to_dict())

(1567, 590) {0: 1463, 1: 104}


## 2. 결측률 50% 초과 피처 제거

In [3]:
missing_ratio = X.isna().mean()
high_missing_cols = missing_ratio[missing_ratio > 0.5].index.tolist()
X = X.drop(columns=high_missing_cols)
print(f"제거된 피처 수: {len(high_missing_cols)} -> 남은 피처 수: {X.shape[1]}")

제거된 피처 수: 28 -> 남은 피처 수: 562


## 3. Train/Test 분리

정보 누출(data leakage) 방지를 위해 imputation/scaling/SMOTE는 train 기준으로 fit.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)
print(y_train.value_counts().to_dict(), y_test.value_counts().to_dict())

(1253, 562) (314, 562)
{0: 1170, 1: 83} {0: 293, 1: 21}


## 4. 결측값 대체 (중앙값)

In [5]:
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

## 5. 저분산(분산 0) 피처 제거

In [6]:
variances = X_train_imp.var()
zero_var_cols = variances[variances == 0].index.tolist()
X_train_imp = X_train_imp.drop(columns=zero_var_cols)
X_test_imp = X_test_imp.drop(columns=zero_var_cols)
print(f"제거된 저분산 피처 수: {len(zero_var_cols)} -> 남은 피처 수: {X_train_imp.shape[1]}")

제거된 저분산 피처 수: 116 -> 남은 피처 수: 446


## 6. StandardScaler 정규화

In [7]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns, index=X_train_imp.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test_imp.columns, index=X_test_imp.index)

## 7. 클래스 불균형 처리 (SMOTE, train만)

In [8]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print("SMOTE 적용 전:", y_train.value_counts().to_dict())
print("SMOTE 적용 후:", y_train_res.value_counts().to_dict())

SMOTE 적용 전: {0: 1170, 1: 83}
SMOTE 적용 후: {0: 1170, 1: 1170}


## 8. 저장

- `X_train.csv` / `y_train.csv`: SMOTE 적용된 학습 데이터 (지도학습용)
- `X_train_raw.csv` / `y_train_raw.csv`: SMOTE 미적용 원본 비율 학습 데이터 (Isolation Forest 등 비지도 모델은 실제 분포로 학습해야 하므로 별도 보관)
- `X_test.csv` / `y_test.csv`: 테스트 데이터 (항상 원본 비율 유지, 평가용)

In [9]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

X_train_res.to_csv(f"{OUT_DIR}/X_train.csv", index=False)
y_train_res.to_csv(f"{OUT_DIR}/y_train.csv", index=False)

X_train_scaled.to_csv(f"{OUT_DIR}/X_train_raw.csv", index=False)
y_train.to_csv(f"{OUT_DIR}/y_train_raw.csv", index=False)

X_test_scaled.to_csv(f"{OUT_DIR}/X_test.csv", index=False)
y_test.to_csv(f"{OUT_DIR}/y_test.csv", index=False)

print("저장 완료:", os.listdir(OUT_DIR))

저장 완료: ['.gitkeep', 'X_test.csv', 'X_train.csv', 'X_train_raw.csv', 'y_test.csv', 'y_train.csv', 'y_train_raw.csv']


## 다음 단계

- `03_modeling.ipynb`: Isolation Forest(비지도, `X_train_raw`로 학습) + XGBoost(지도, `X_train`으로 학습) 앙상블